# Multi Document Summurizer

In [19]:
!pip install summa pypdf torch transformers

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
    --------------------------------------- 1.8/114.6 MB 10.3 MB/s eta 0:00:11
   - -------------------------------------- 4.5/114.6 MB 11.4 MB/s eta 0:00:10
   -- ------------------------------------- 6.6/114.6 MB 10.7 MB/s eta 0:00:11
   --- ------------------------------------ 8.7/114.6 MB 10.5 MB/s eta 0:00:11
   --- ------------------------------------ 10.7/114.6 MB 10.5 MB/s eta 0:00:10
   ---- ----------------------------------- 13.1/114.6 MB 10.5 MB/s eta 0:00:10
   ----- ---------------------------------- 15.2/114.6 MB 10.5 MB/s eta 0:00:10
   ------ --------------------------------- 17.6/114.6 MB 10.5 MB/s eta 0:00:10
   ------ --------------------------------- 19.9/114.6 MB 10.6 MB/s eta 0:00:09
   ------- -------------------------------- 22.3/114.6 MB 10.7 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.2 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Part 1: Extractive Summarization
Use summa to extract top sentences from each document separately.

In [6]:
from summa import summarizer
from pypdf import PdfReader

In [7]:
# files = input("Enter the file names (space separated): ").split(" ")
files = ["one.pdf", "two.pdf", "three.pdf"]
print("Files entered:", files)

Files entered: ['one.pdf', 'two.pdf', 'three.pdf']


In [13]:
def summarize_extract(file):
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    summary = summarizer.summarize(text, ratio=0.1)
    return summary

In [14]:
extractive_summaries = []
for file in files:
    summary = summarize_extract(file)
    print(f"Summary for {file}:\n{summary}\n")
    extractive_summaries.append(summary)

Summary for one.pdf:
This paper describes LL VM (Low Level Virtual Machine),
LL VM deﬁnes a common, low-level
implement high-level language features; an instruction fo r
The LL VM compiler framework and code
scribe the design of the LL VM representation and compiler
type information it provides; (b) compiler performance for
ples of the beneﬁts LL VM provides for several challenging
This paper presents LL VM — Low-Level Virtual Ma-
LL VM achieves this through two parts: (a) a code rep-
The LL VM code representation describes a program using
novel features in the LL VM code representation: (a) A low-
level, language-independent type system that can be used to
The LL VM representation issource-language-independent,for two reasons.
representing code with little type information.
that LL VM is not intended to be a universal compiler IR .
In particular, LL VM does not represent high-level language
Because of the diﬀering goals and representations, LL VM
First, LL VM has no notion of high-
Se

Multiple definitions in dictionary at byte 0x432f for key /ToUnicode


Summary for two.pdf:
This paper describes LL VM (Low Level Virtual Machine),
LL VM deﬁnes a common, low-level
implement high-level language features; an instruction fo r
The LL VM compiler framework and code
scribe the design of the LL VM representation and compiler
type information it provides; (b) compiler performance for
ples of the beneﬁts LL VM provides for several challenging
This paper presents LL VM — Low-Level Virtual Ma-
LL VM achieves this through two parts: (a) a code rep-
The LL VM code representation describes a program using
novel features in the LL VM code representation: (a) A low-
level, language-independent type system that can be used to
The LL VM representation issource-language-independent,for two reasons.
representing code with little type information.
that LL VM is not intended to be a universal compiler IR .
In particular, LL VM does not represent high-level language
Because of the diﬀering goals and representations, LL VM
First, LL VM has no notion of high-
Se

In [17]:
len(extractive_summaries[0])

9258

## Part 2: Abstraction summarization
In this phase the summaries extracted form each document are combined and summarised together using distilBart-12-6 model.

In [20]:
from transformers import AutoTokenizer, BartForConditionalGeneration

c:\Users\mrz\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

c:\Users\mrz\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrz\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weight

In [27]:
finalInput = ""
for extSum in extractive_summaries:
    inputs = tokenizer(extSum, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=512, early_stopping=True)
    abstractive_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    finalInput += abstractive_summary + " [SEP] "
print(f"Final Abstractive Summary:\n{finalInput}\n")

Final Abstractive Summary:
 This paper describes LL VM (Low Level Virtual Machine), a low-level, language-independent type system that can be used to implement high-level language features . LL VM is not intended to be a universal compiler IR . It does not specify a specific type system can be implemented in LL VM itself . The LL VM program may not be reliable . [SEP]  This paper describes LL VM (Low Level Virtual Machine), a low-level, language-independent type system that can be used to implement high-level language features . LL VM is not intended to be a universal compiler IR . It does not specify a specific type system can be implemented in LL VM itself . The LL VM program may not be reliable . [SEP]  In Rust or Rust, Rust or conversion between integers and pointers in C or C++, we explain the memory model currently used by LLVM . We use int to represent an integer type that is sufficiently large to hold a pointer for brevity . We track a set of addresses that have to be within bo

In [28]:
inputs = tokenizer(finalInput, return_tensors="pt")
summary_ids = model.generate(inputs["input_ids"], num_beams=4, early_stopping=True)
finalSummary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Final Summary:\n{finalSummary}\n")

Final Summary:
 This paper describes LL VM (Low Level Virtual Machine), a low-level, language-independent type system that can be used to implement high-level language features . LL VM is not intended to be a universal compiler IR . It does not specify a specific type system can be implemented in LL VM itself . The LL VM program may not be reliable .

